# 이커머스 멀티모달 분석 보고서

B1~B4의 데이터 연결과 피처 생성을 구현했다. 정제·RFM·차트는 통합 계획의 C~D 단계에서 이어서 작성한다.

표본은 비공개 Drive에서 복원한다. 이 노트북의 실행 출력에는 고객 ID와 거래 행이 포함될 수 있으므로 공개 저장본은 출력을 비운다. `python -m utils.run_checks --with-sample`은 현재 가상환경의 새 커널로 실행하고 결과를 Git에서 제외된 `data/processed/`에 저장한다.

## 설정

경로·날짜·시드·이미지 크기·배치 크기는 여기에서 정한다. 추출 기록과 다른 설정으로 분석하려면 먼저 표본을 다시 추출한다.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src" / "pipeline.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "pipeline.py").is_file():
    raise RuntimeError("저장소 루트 또는 notebooks 폴더에서 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import DataAnalyzer

DATA = ROOT / "data" / "raw" / "hm"
SETTINGS = {"start_inclusive": "2020-06-01", "end_exclusive": "2020-09-01",
            "seed": 42, "sales_channel": None, "image_size": 32, "batch_size": 32}
manifest = json.loads((DATA / "sample_manifest.json").read_text(encoding="utf-8"))
for setting in ("start_inclusive", "end_exclusive", "seed", "sales_channel"):
    assert manifest[setting] == SETTINGS[setting], f"표본 설정 불일치: {setting}"
print("Python:", sys.version.split()[0])
print("표본의 기준일:", manifest["reference_date"])

## NB_LOAD · 실제 거래와 상품 연결

H&M의 원본 열 이름을 유지한다. `t_dat`는 구매일, `price`는 원본 구매금액 단위, `prod_name`은 상품명이다. ID는 문자열로 읽고 날짜 변환 실패는 `NaT`로 남긴다. 잘못된 거래의 분리와 금액 검증은 C1에서 구현한다.

상품 키가 중복이면 조인을 중단한다. 빈 상품 키는 제외하며, 미매칭 거래는 유지해 별도로 센다. 조인 전후 행 수와 금액 합계를 확인하고 합계 비교에는 `rtol=1e-12`, `atol=1e-10`을 적용한다. 아래 원본 예시는 로컬 비공개 출력이다.

In [ ]:
analyzer = DataAnalyzer(DATA / "transactions_sample.csv")
transactions = analyzer.load_data(
    required_columns=["customer_id", "article_id", "t_dat", "price", "sales_channel_id"],
    date_columns=["t_dat"])
assert len(transactions) == manifest["transactions"]["sampled_rows"]
assert transactions["customer_id"].nunique() == manifest["sampled_customers"]
assert transactions["t_dat"].between(pd.Timestamp(SETTINGS["start_inclusive"]),
    pd.Timestamp(SETTINGS["end_exclusive"]), inclusive="left").all()
joined = analyzer.join_products(DATA / "articles_sample.csv")
joined_column_count = joined.shape[1]
assert len(joined) >= 1000 and joined.shape[1] >= 8
assert analyzer.join_summary["rows_before"] == analyzer.join_summary["rows_after"]
assert np.isclose(analyzer.join_summary["amount_before"], analyzer.join_summary["amount_after"],
                  rtol=1e-12, atol=1e-10)
assert analyzer.products["article_id"].nunique() == manifest["matched_products"]
display(pd.DataFrame([analyzer.join_summary]))
display(joined.head())
joined.info()
display(joined.describe(include="all"))

## NB_FEATURES · 이미지와 텍스트 수치화

상품별로 한 번 계산한 피처를 거래에 연결한다. 거래가 많은 상품을 상품 EDA에서 중복 집계하지 않도록 `analyzer.products`도 유지한다.

이미지는 RGB로 맞춘 뒤 정수 픽셀을 0~1로 정규화하고 NumPy 인덱싱으로 32×32로 줄인다. 파일 입출력은 상품별로 수행하며, 통계는 `(N,H,W,C)` 배치의 `axis=(1,2,3)`에서 계산한다. 표준편차는 모집단 기준 `ddof=0`이다. 피처는 축소 이미지의 전체 채널 평균과 표준편차로, 원본 해상도의 통계와 다를 수 있다.

`ok`, `missing`, `not_sampled`, `corrupt`, `too_small`로 상태를 기록한다. 상품 미매칭 거래는 `unmatched_product`다. 실패 피처를 0으로 채우지 않는다. 텍스트는 결측 수를 먼저 기록하고 계산할 때만 빈 문자열로 취급한다.

In [ ]:
missing_product_names = int(analyzer.products["prod_name"].isna().sum())
analyzer.extract_text_features("prod_name")
analyzer.extract_image_features(image_dir=DATA / "images",
    image_size=SETTINGS["image_size"], batch_size=SETTINGS["batch_size"])
products = analyzer.products
image_statuses = products["image_status"].value_counts().to_dict()
assert image_statuses.get("ok", 0) == manifest["sampled_images"]
assert image_statuses.get("missing", 0) == manifest["matched_products_without_images"]
assert image_statuses.get("not_sampled", 0) == manifest["products_with_images_outside_image_sample"]
assert products.loc[products["image_status"] != "ok", ["image_mean", "image_std"]].isna().all().all()
assert analyzer.df.shape[0] == manifest["transactions"]["sampled_rows"]
assert analyzer.df.columns.is_unique
print("원래 결측 상품명 수:", missing_product_names)
display(pd.Series(image_statuses, name="상품 수").to_frame())
display(products[["prod_name_length", "prod_name_word_count", "image_mean", "image_std"]].describe())
print("가장 큰 실제 배치:", analyzer.image_batch_info)
# 이 값은 배치 배열 자체의 메모리다. 원본 디코딩·중간 배열은 별도로 필요하다.
print("설정상 최대 배치 배열(bytes):", SETTINGS["batch_size"] * SETTINGS["image_size"] ** 2 * 3 * 4)

### 알려진 입력으로 계산 확인

가상 상품명 `Blue cotton shirt`는 17자·3단어다. 픽셀 절반이 0이고 절반이 1인 배열의 평균과 표준편차는 각각 0.5다. 회색조·RGB·RGBA·손상·누락·크기 미달·배치 크기 변경 검증은 `tests/test_pipeline.py`에서 재현한다.

In [ ]:
fixture = DataAnalyzer("unused.csv")
fixture.df = pd.DataFrame({"prod_name": ["Blue cotton shirt"],
                          "image": [np.array([[0, 0], [1, 1]], dtype=np.float32)]})
fixture.extract_text_features("prod_name")
fixture.extract_image_features("image")
assert fixture.df["prod_name_length"].iloc[0] == 17
assert fixture.df["prod_name_word_count"].iloc[0] == 3
np.testing.assert_allclose(fixture.df[["image_mean", "image_std"]].to_numpy(), .5)
display(fixture.df.drop(columns="image"))

### 필수 메서드 연결 점검

아래 가상 거래는 로드·결측 처리·IQR·RFM 메서드가 노트북에서 호출되는지 확인한다. 실제 거래의 정제 정책과 RFM 정의 검증은 C1~C5에서 수행한다. 현재 RFM의 최종일+1·행 수 빈도·동점 순위 문제는 통합 계획에 기록되어 있다.

In [ ]:
with tempfile.TemporaryDirectory(dir=ROOT / "data") as temporary:
    path = Path(temporary) / "api_fixture.csv"
    pd.DataFrame({"customer_id": list("ABCDE"), "t_dat": ["2020-08-01"] * 5,
                  "price": [1., 2., 3., 4., 100.]}).to_csv(path, index=False)
    api = DataAnalyzer(path)
    api.load_data(date_columns=["t_dat"])
    api.handle_missing_values(strategy="mean")
    flags = api.detect_outliers("price", threshold=1.5)
    assert flags.tolist() == [False, False, False, False, True]
    rfm_api = api.calculate_rfm("customer_id", "t_dat", "price")
    assert len(rfm_api) == 5 and {"Recency", "Frequency", "Monetary", "Segment"} <= set(rfm_api)
print("필수 메서드 4개 호출 완료. 실제 RFM 계산 검증은 C 단계에서 진행.")

### 로컬 검증 기록

실데이터의 수치와 실행 출력은 `data/processed/`에만 저장한다. 공개 Git에는 코드와 가상 입력의 기대값을 남긴다.

In [ ]:
summary = {"stage": "B1-B4", "settings": SETTINGS,
    "reference_date": manifest["reference_date"], "join": analyzer.join_summary,
    "transaction_rows": len(analyzer.df), "joined_columns": joined_column_count,
    "feature_columns": list(analyzer.df.columns), "products": len(products),
    "missing_product_names": missing_product_names,
    "image_statuses": {str(key): int(value) for key, value in image_statuses.items()},
    "largest_batch": analyzer.image_batch_info,
    "fixture": {"text_length": 17, "word_count": 3, "image_mean": .5, "image_std": .5}}
output = ROOT / "data" / "processed" / "b1_b4_verification.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("B1~B4 연결 및 피처 검증 완료. 기록:", output.relative_to(ROOT))

## NB_CLEAN · 결측과 이상치

C1~C3에서 유효 거래 분리, 지정 피처 대치와 IQR 전후 비교를 구현한다. 아직 미완료다.

## NB_STATS · 통계와 상관계수

D2에서 분석 단위를 구분해 통계와 상관계수 2쌍 이상을 해석한다. 아직 미완료다.

## NB_PLOTS · 6종 차트

D3에서 제목·축 레이블·단위를 갖춘 필수 차트를 만든다. 아직 미완료다.

## NB_RFM · 고객 세분화

C4~C5와 D4에서 수정한 RFM 정의와 세그먼트 프로파일을 검증한다. 아직 미완료다.

## NB_REPRO · 제출 재현

F1에서 새 복제본의 전체 실행과 최종 집계 일치를 확인한다. 아직 미완료다.